<a href="https://colab.research.google.com/github/arun-antony/ai-colab-tryout/blob/main/tcs_ai_system_case_study_stock_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. Import Dependencies

In [55]:
import pandas as pd
import numpy as np
from pathlib import Path

### B. Set Paths

In [59]:
PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data"
EXPORT_DIR = PROJECT_DIR / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

# Define the path to the stock transaction Excel file
STOCK_ACTIVITY_FILE_PATH = DATA_DIR / "robinhood_style_stock_activity_case_study.xlsx"

if not STOCK_ACTIVITY_FILE_PATH.exists():
    raise FileNotFoundError(f"File not found: {STOCK_ACTIVITY_FILE_PATH}")

### C. Helper Methods

In [79]:
# Exporting df as CSV
def export_df(df, filename):
    export_path = EXPORT_DIR / filename
    df.to_csv(export_path, index=False)
    print(f"Exported data frame for reference to: {export_path}")

## Start of Epic 1: Data Engineering & Feature Foundation

### 1. Data Ingestion
Jira Ticket: https://tcstradingassistant.atlassian.net/browse/DYRPTA-12

In [81]:
# Read the Excel file into a pandas DataFrame
df = pd.read_excel(STOCK_ACTIVITY_FILE_PATH)

# Display the first 5 rows of the DataFrame
display(df.head())

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount
0,2021-01-05,2021-01-05,2021-01-05,NaN,ACH Deposit,ACH,NaN,NaN,2500.00
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,Buy,4.6606,206.74,-963.53
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,Buy,1.5733,200.01,-314.68
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,Buy,10.6469,98.73,-1051.17
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,Buy,4.3080,254.32,-1095.61


### 2. Data Cleaning / Sanitization
Jira: https://tcstradingassistant.atlassian.net/browse/DYRPTA-15

Following cleanups are done in this section:
1. Remove all rows that are not a trading transaction. Eg: Deposit, Interest payout etc. Create a filter list that decides what kind of transaction to be filtered.
2. Remove transactions that does not have a date or date is invalid
3. Remove transactions that does not have a quantity
4. Remove transactions that does not have a price




In [3]:
#@title Flags that control what kind of data rows need to be removed or cleaned { run: "auto" }
apply_transaction_code_filter = True #@param {type:"boolean"}
apply_invalid_date_filter = True #@param {type:"boolean"}
apply_invalid_quantity_filter = True #@param {type:"boolean"}
apply_invalid_price_filter = True #@param {type:"boolean"}

In [78]:
# Create a copy to avoid SettingWithCopyWarning
cleaned_df = df.copy()

if apply_transaction_code_filter:
    # 1.1 Removes rows where 'Trans Code' is missing completely
    cleaned_df = cleaned_df.dropna(subset=['Trans Code'])

    # List of transaction codes that will be allowedd for processing
    # Note: Use lower cased values in this filter list
    transaction_code_allow_filter = ['buy', 'sell']

    # 1.2 Apply filter so that only allowed transactions are retained
    cleaned_df = cleaned_df[cleaned_df['Trans Code'].str.lower().isin(transaction_code_allow_filter)]

    # lower case all the tranaction codes
    cleaned_df['Trans Code'] = cleaned_df['Trans Code'].str.lower()

if apply_invalid_date_filter:
    date_columns = ['Activity Date', 'Process Date', 'Settle Date']
    # 2.1 Drop rows where any of the date is NaN
    cleaned_df = cleaned_df.dropna(subset=date_columns, how='any')

    # 2.2 Convert all 3 columns to datetime, turning unparseable dates into NaT (Not a Time)
    for col in date_columns:
        cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

    # 2.3 Drop rows again if the conversion generated any new NaT values
    cleaned_df = cleaned_df.dropna(subset=date_columns, how='any')

if apply_invalid_quantity_filter:
    # 3.1 Drop rows where 'Quantity' is NaN
    cleaned_df = cleaned_df.dropna(subset=['Quantity'])

if apply_invalid_price_filter:
    # 4.1 Drop rows where 'Price' is NaN
    cleaned_df = cleaned_df.dropna(subset=['Price'])


cleaned_df.head(10)

,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,buy,4.6606,206.74,-963.53
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,buy,1.5733,200.01,-314.68
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,buy,10.6469,98.73,-1051.17
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,buy,4.3080,254.32,-1095.61
5,2021-03-10,2021-03-10,2021-03-12,PLTR,Market Buy 5.54 shares of Palantir Technologie...,buy,5.5400,15.03,-83.27
6,2021-03-15,2021-03-15,2021-03-17,AMD,Market Buy 8.8108 shares of Advanced Micro Dev...,buy,8.8108,81.08,-714.38
7,2021-03-20,2021-03-20,2021-03-22,AMD,Market Sell 8.8108 shares of Advanced Micro De...,sell,-8.8108,84.47,744.25
8,2021-03-21,2021-03-21,2021-03-23,MSFT,Market Sell 4.308 shares of Microsoft Corporation,sell,-4.3080,237.07,1021.30
9,2021-03-24,2021-03-24,2021-03-26,GOOGL,Market Buy 11.1618 shares of Alphabet Inc.,buy,11.1618,88.07,-983.02
11,2021-04-04,2021-04-04,2021-04-06,MSFT,Market Sell 1.5733 shares of Microsoft Corpora...,sell,-1.5733,208.67,328.30


### 3. Core Metric Engineering
Jira Ticket: https://tcstradingassistant.atlassian.net/browse/DYRPTA-13

The following metrics are calculated in this funtion:


1. Total Cost Basis
2. Per Unit Cost Basis
3. Realized PnL
4. Result Percentage
5. Holding Period (Weighted)
6. Holding Period (As a dictionary showing the exact split)
7. Win or Loss

In [71]:
# Constants representing the metrics

# Field Names
PER_UNIT_COST_BASIS = 'per_unit_cost'
TOTAL_COST_BASIS = 'total_cost'
REALIZED_PNL = 'realized_pnl'
RESULT_PERCENTAGE = 'return_percentage'
AVERAGE_HOLDING_PERIOD = 'average_holding_days'
HOLDING_DETAILS = 'holding_details'
WIN_LOSS_STATUS = 'win_loss_status'

# Field Values
WIN = 'Win'
LOSS = 'Loss'
FLAT = 'Flat'
NOT_APPLICABLE = 'NA'

In [76]:
# Component of the Core Metric Extraction for BUY transaction
def _process_buy_transaction(qty, price, activity_date, ticker, inventory):
    """Adds shares to inventory and returns blank performance metrics."""
    inventory[ticker].append({
        'qty': qty,
        'price': price,
        'act_date': activity_date
    })

    return {
        PER_UNIT_COST_BASIS: price,
        TOTAL_COST_BASIS: price * qty,
        REALIZED_PNL: np.nan,
        RESULT_PERCENTAGE: np.nan,
        AVERAGE_HOLDING_PERIOD: np.nan,
        HOLDING_DETAILS: [],
        WIN_LOSS_STATUS: NOT_APPLICABLE
    }

In [72]:
# Component of the Core Metric Extraction for SELL transaction
def _process_sell_transaction(qty, price, activity_date, ticker, inventory):
    """Executes FIFO queue matching and calculates realized performance."""
    shares_to_sell = qty
    total_cost_of_shares_sold = 0.0
    total_weighted_holding_days = 0.0
    lot_breakdown = []

    while shares_to_sell > 0 and len(inventory.get(ticker, [])) > 0:
        current_lot = inventory[ticker][0]
        lot_qty, lot_price = current_lot['qty'], current_lot['price']
        days_held = (activity_date - current_lot['act_date']).days

        if lot_qty <= shares_to_sell:
            total_cost_of_shares_sold += lot_qty * lot_price
            total_weighted_holding_days += lot_qty * days_held
            lot_breakdown.append({'qty': lot_qty, 'days': days_held})

            shares_to_sell -= lot_qty
            inventory[ticker].pop(0)
        else:
            total_cost_of_shares_sold += shares_to_sell * lot_price
            total_weighted_holding_days += shares_to_sell * days_held
            current_lot['qty'] = round(current_lot['qty'] - shares_to_sell, 4)
            lot_breakdown.append({'qty': shares_to_sell, 'days': days_held})
            shares_to_sell = 0

    # Fallback for unmatched shares
    if shares_to_sell > 0:
        total_cost_of_shares_sold += shares_to_sell * price
        lot_breakdown.append({'qty': shares_to_sell, 'days': 0, 'note': 'unmatched'})

    # Compute final metrics
    realized_pnl = (qty * price) - total_cost_of_shares_sold
    return_pct = (100 * realized_pnl / total_cost_of_shares_sold) if total_cost_of_shares_sold > 0 else 0.0

    return {
        PER_UNIT_COST_BASIS: total_cost_of_shares_sold / qty,
        TOTAL_COST_BASIS: total_cost_of_shares_sold,
        REALIZED_PNL: realized_pnl,
        RESULT_PERCENTAGE: return_pct,
        AVERAGE_HOLDING_PERIOD: total_weighted_holding_days / qty,
        HOLDING_DETAILS: lot_breakdown,
        WIN_LOSS_STATUS: WIN if realized_pnl > 0 else LOSS if realized_pnl < 0 else FLAT
    }

In [74]:
# Component of the Core Metric Extraction that orchestrates the calculations
def derive_core_metrics(df):
    processed_df = df.sort_values(by=['Activity Date', 'Process Date']).copy()
    inventory = {}

    # We will collect the dictionaries returned by the helpers here
    results_map = {}

    for idx, row in processed_df.iterrows():
        ticker = row['Instrument']
        trans_type = row['Trans Code'].strip().lower()
        qty = abs(float(row['Quantity']))
        price = float(row['Price'])
        activity_date = row['Activity Date']

        if ticker not in inventory:
            inventory[ticker] = []

        if trans_type == 'buy':
            results_map[idx] = _process_buy_transaction(qty, price, activity_date, ticker, inventory)
        elif trans_type == 'sell':
            results_map[idx] = _process_sell_transaction(qty, price, activity_date, ticker, inventory)
        else:
            # Create a blank fallback dictionary
            results_map[idx] = {PER_UNIT_COST_BASIS: 0.0, TOTAL_COST_BASIS: 0.0, REALIZED_PNL: 0.0,
                                RESULT_PERCENTAGE: 0.0, AVERAGE_HOLDING_PERIOD: np.nan,
                                HOLDING_DETAILS: [], WIN_LOSS_STATUS: NOT_APPLICABLE}

    # Map the collected dictionary keys to new columns efficiently
    for key in [PER_UNIT_COST_BASIS, TOTAL_COST_BASIS, REALIZED_PNL, RESULT_PERCENTAGE, AVERAGE_HOLDING_PERIOD, HOLDING_DETAILS, WIN_LOSS_STATUS]:
        processed_df[key] = processed_df.index.map(lambda i: results_map[i][key])

    return processed_df

###4. Strategy Classification Pipeline

Jira: https://tcstradingassistant.atlassian.net/browse/DYRPTA-14

Label trades (Short-term, Swing, Long-term) based on Holding Period thresholds.

In [93]:
def _generate_trade_labels(row):
    """
    Evaluates raw trade metrics and returns categorical labels
    for performance and trading style.
    """
    labels = {}
    holding_days = row[AVERAGE_HOLDING_PERIOD]

    # 1. Trading Style / Duration Label
    if pd.isna(holding_days):
        labels['holding_style'] = 'Open Position'
    elif holding_days == 0:
        labels['holding_style'] = 'Day Trade'
    elif 1 <= holding_days <= 30:
        labels['holding_style'] = 'Swing Trade'
    elif 31 <= holding_days <= 365:
        labels['holding_style'] = 'Position Trade'
    else:
        labels['holding_style'] = 'Long-Term Investment'

    # 2. Capital Gains Tax Bracket (US Standard)
    if pd.isna(holding_days):
        labels['tax_bracket'] = 'NA'
    elif holding_days <= 365:
        labels['tax_bracket'] = 'Short-Term'
    else:
        labels['tax_bracket'] = 'Long-Term'

    return pd.Series({
        'holding_style': labels['holding_style'],
        'tax_bracket': labels['tax_bracket']
    })


In [94]:
def apply_trade_labels(df):
    """
    Post-processing function that reads calculated core metrics
    and applies behavioral, style, and tax labels across the DataFrame.
    """
    # Create a copy to prevent SettingWithCopyWarning
    labeled_df = df.copy()


    # Apply the labeling logic across the rows (axis=1)
    label_cols = labeled_df.apply(_generate_trade_labels, axis=1)

    # Merge the new label columns back into our main DataFrame
    labeled_df = pd.concat([labeled_df, label_cols], axis=1)

    return labeled_df

### 5. End of Epic 1: Data Engineering & Feature Foundation

Gets the df with clean data and with features extracted into additional columns. Look for the columns:
1. per_unit_cost,
2. total_cost,
3. realized_pnl,
4. return_percentage,
5. holding_days,
6. holding_details,
7. win_loss_status,
8. holding style
9. tax bracket

In [95]:
df_with_metrics = derive_core_metrics(cleaned_df.copy())
df_with_labels = apply_trade_labels(df_with_metrics.copy())

df_with_labels.head()


,Activity Date,Process Date,Settle Date,Instrument,Description,Trans Code,Quantity,Price,Amount,per_unit_cost,total_cost,realized_pnl,return_percentage,average_holding_days,holding_details,win_loss_status,holding_style,tax_bracket
1,2021-01-18,2021-01-18,2021-01-20,META,Market Buy 4.6606 shares of Meta Platforms Inc.,buy,4.6606,206.74,-963.53,206.74,963.532444,NaN,NaN,NaN,[],NA,Open Position,NA
2,2021-02-03,2021-02-03,2021-02-05,MSFT,Market Buy 1.5733 shares of Microsoft Corporation,buy,1.5733,200.01,-314.68,200.01,314.675733,NaN,NaN,NaN,[],NA,Open Position,NA
3,2021-02-28,2021-02-28,2021-03-02,AMD,Market Buy 10.6469 shares of Advanced Micro De...,buy,10.6469,98.73,-1051.17,98.73,1051.168437,NaN,NaN,NaN,[],NA,Open Position,NA
4,2021-03-07,2021-03-07,2021-03-09,MSFT,Market Buy 4.308 shares of Microsoft Corporation,buy,4.3080,254.32,-1095.61,254.32,1095.610560,NaN,NaN,NaN,[],NA,Open Position,NA
5,2021-03-10,2021-03-10,2021-03-12,PLTR,Market Buy 5.54 shares of Palantir Technologie...,buy,5.5400,15.03,-83.27,15.03,83.266200,NaN,NaN,NaN,[],NA,Open Position,NA


In [97]:
export_df(df_with_labels, 'epic_1_output.csv')

Exported data frame for reference to: exports/epic_1_output.csv


### Start Epic 2 from here

In [ ]:
# you can start with df_with_labels